# Coding Agent from scratch — Together AI + Eval

This notebook uses your **existing uv project** and Jupyter kernel. No `mini-swe-agent` framework, no fake LLM.

Install in your existing project's terminal: `uv add together`. Select the project's Python kernel in JupyterLab.

Set `TOGETHER_API_KEY` in the environment *before launching JupyterLab*. Avoid storing API keys in notebooks.

We will build a 3-tool coding agent (read, write, test) and then evaluate it using unseen tests. Execution uses a local subprocess, **not a security sandbox**; use only for controlled exercises. We deliberately do not grant arbitrary shell access.

In [1]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv
from together import Together

load_dotenv(Path.cwd() / ".env", override=True)

api_key = os.getenv("TOGETHER_API_KEY")
assert api_key, "TOGETHER_API_KEY not found"

client = Together(api_key=api_key)

print("Together API ready!")

Together API ready!


In [2]:
response = client.chat.completions.create(
    model="Qwen/Qwen3.5-9B",
    messages=[
        {"role": "user", "content": "What is 1 + 1?"}
    ],
)

print(response.choices[0].message.content)

1 + 1 = 2.


## 1 — Create a buggy coding task

A scratch directory is created automatically. This does **not** create another uv project or modify your existing code. Public tests are visible to the agent, hidden tests will not be exposed.

In [3]:
workspace = Path.cwd() / "agent-tutorial"
workspace.mkdir(parents=True, exist_ok=True)

(workspace / "solution.py").write_text("""def median(values):
    values = sorted(values)
    if not values:
        raise ValueError('empty input')
    return values[len(values) // 2]
""", encoding="utf-8")

(workspace / "test_public.py").write_text("""import unittest
from solution import median

class TestMedian(unittest.TestCase):
    def test_odd(self):
        self.assertEqual(median([3, 1, 2]), 2)

    def test_even(self):
        self.assertEqual(median([1, 3, 5, 7]), 4)
""", encoding="utf-8")

print("Scratch workspace:", workspace)
print((workspace / "solution.py").read_text())

Scratch workspace: /Users/yuqingpeng/Documents/fighting-rl/eval/agent-tutorial
def median(values):
    values = sorted(values)
    if not values:
        raise ValueError('empty input')
    return values[len(values) // 2]



## 2 — Build the evaluator first

We run tests in a subprocess so that a crash does not terminate Jupyter. **This is not process or filesystem isolation**. We strip API credentials from the child process environment. Later we can replace this with Docker.

In [4]:
def run_tests(folder: Path, pattern: str) -> dict:
    proc = subprocess.run(
        [sys.executable, "-m", "unittest", "discover", "-s", str(folder), "-p", pattern, "-v"],
        cwd=folder,
        env={"PATH": os.environ.get("PATH", ""), "PYTHONIOENCODING": "utf-8"},
        capture_output=True,
        text=True,
        timeout=10,
    )
    return {"passed": proc.returncode == 0, "output": (proc.stdout + proc.stderr)[-3000:]}

baseline = run_tests(workspace, "test_public.py")
print("Baseline passed:", baseline["passed"])
print(baseline["output"])

Baseline passed: False
test_even (test_public.TestMedian.test_even) ... FAIL
test_odd (test_public.TestMedian.test_odd) ... ok

FAIL: test_even (test_public.TestMedian.test_even)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/Users/yuqingpeng/Documents/fighting-rl/eval/agent-tutorial/test_public.py", line 9, in test_even
    self.assertEqual(median([1, 3, 5, 7]), 4)
AssertionError: 5 != 4

----------------------------------------------------------------------
Ran 2 tests in 0.000s

FAILED (failures=1)



## 3 — Define tools

The LLM can only ask to read two known files, overwrite `solution.py`, or run public tests. These are **our tools**, not `mini-swe-agent` tools.

In [5]:
TOOLS = [
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file in the task workspace.",
        "parameters": {"type": "object", "properties": {
            "filename": {"type": "string", "enum": ["solution.py", "test_public.py"]}},
            "required": ["filename"], "additionalProperties": False}}},
    
    {"type": "function", "function": {
        "name": "write_solution", "description": "Replace solution.py with complete Python code.",
        "parameters": {"type": "object", "properties": {
            "content": {"type": "string"}},
            "required": ["content"], "additionalProperties": False}}},
    
    {"type": "function", "function": {
        "name": "run_public_tests", "description": "Run tests visible to the agent.",
        "parameters": {"type": "object", "properties": {}, "additionalProperties": False}}},
]

def execute_tool(name: str, args: dict) -> dict:
    if name == "read_file":
        filename = args["filename"]
        
        if filename not in {"solution.py", "test_public.py"}:
            raise ValueError("File not allowed")
            
        return {"content": (workspace / filename).read_text(encoding="utf-8")}
        
    if name == "write_solution":
        content = args["content"]
        
        if not isinstance(content, str) or len(content) > 20000:
            raise ValueError("Invalid content")
            
        (workspace / "solution.py").write_text(content, encoding="utf-8")
        return {"status": "written"}
        
    if name == "run_public_tests":
        return run_tests(workspace, "test_public.py")
        
    raise ValueError(f"Unknown tool: {name}")

## 4 — Real agent loop

Run this cell to call the real LLM and execute tools. Instead of dumping output on every call, it stores a structured trace in `messages` and per-request token counts in `usage_by_step`. Expand the trace in the **next cell** to inspect what happened. No agent framework.


In [14]:
MODEL = "Qwen/Qwen3.5-9B"

SYSTEM = """You are a coding agent. Your job is to fix solution.py.
Tools available: read_file, write_solution, run_public_tests.
Read source and tests first, implement a general solution, and run public tests.
Do not hardcode individual test values. When done, give a short final answer.
"""

messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "Fix median() for any list of numbers, preserving the ValueError for empty lists."},
]

MAX_STEPS = 12
usage_by_step = []
stop_reason = "max_steps"

for step in range(1, MAX_STEPS + 1):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
        max_tokens=2000,
    )

    # Track both inputs and outputs for every API request.
    usage = response.usage
    usage_by_step.append({
        "step": step,
        "input": getattr(usage, "prompt_tokens", None),
        "output": getattr(usage, "completion_tokens", None),
        "total": getattr(usage, "total_tokens", None),
    })

    msg = response.choices[0].message
    calls = msg.tool_calls or []
    assistant_message = {"role": "assistant", "content": msg.content or ""}

    if calls:
        assistant_message["tool_calls"] = [
            {"id": call.id, "type": "function", "function": {
                "name": call.function.name, "arguments": call.function.arguments}}
            for call in calls
        ]

    messages.append(assistant_message)
    if not calls:
        stop_reason = "model_finished"
        break

    for call in calls:
        try:
            args = json.loads(call.function.arguments or "{}")
            result = execute_tool(call.function.name, args)
        except Exception as error:
            result = {"error": f"{type(error).__name__}: {error}"}

        # Record the complete result; the viewer controls what is displayed.
        messages.append({
            "role": "tool", "tool_call_id": call.id,
            "content": json.dumps(result, ensure_ascii=False),
        })

print(f"Run complete: {stop_reason}; {len(usage_by_step)} LLM turns. Expand the trace below.")


Run complete: model_finished; 5 LLM turns. Expand the trace below.


## Format the traces

See the redundant tool call!

In [15]:
import json
from html import escape
from itertools import groupby
from IPython.display import HTML, display


def show_agent_trace(messages, usage_by_step):
    """Show readable, expandable tool calls while retaining the full trace."""
    results = {}
    for msg in messages:
        if msg.get("role") == "tool":
            try:
                results[msg["tool_call_id"]] = json.loads(msg["content"])
            except (ValueError, TypeError):
                results[msg["tool_call_id"]] = {"raw": msg.get("content", "")}

    turns = [msg for msg in messages if msg.get("role") == "assistant"]
    n_calls = sum(len(msg.get("tool_calls", [])) for msg in turns)
    n_input = sum((row["input"] or 0) for row in usage_by_step)
    n_output = sum((row["output"] or 0) for row in usage_by_step)
    complete_usage = all(row["input"] is not None and row["output"] is not None for row in usage_by_step)

    # Public list prices for openai/gpt-oss-120b on Together, in USD per million tokens.
    estimated_cost = (n_input * 0.15 + n_output * 0.60) / 1_000_000
    usage_note = "approximate" if complete_usage else "incomplete (missing usage from at least one call)"

    lines = [
        "<h3>Agent Trace</h3>",
        f"<p><b>{len(turns)}</b> LLM turns · <b>{n_calls}</b> tool calls · "
        f"<b>{n_input:,}</b> input tokens · <b>{n_output:,}</b> output tokens · "
        f"estimated cost <b>${estimated_cost:.5f}</b> ({escape(usage_note)})</p>",
    ]

    for i, turn in enumerate(turns):
        calls = turn.get("tool_calls", [])
        usage = usage_by_step[i] if i < len(usage_by_step) else {}
        intro = escape(" ".join((turn.get("content") or "").split())[:140])
        lines.append(
            f"<details open><summary><b>Turn {i+1}</b> · {len(calls)} calls · "
            f"input {usage.get('input', '?')}, output {usage.get('output', '?')} · {intro}</summary>"
        )

        events = []
        for call in calls:
            name = call["function"]["name"]
            raw = call["function"].get("arguments") or "{}"
            try:
                args = json.loads(raw)
            except (TypeError, ValueError):
                args = None
            result = results.get(call["id"], {"error": "Missing tool result"})

            label = name
            if name == "read_file":
                label = f"read_file({args.get('filename', '?') if isinstance(args, dict) else '?'})"
            if args is None:
                status = "INVALID JSON"
            elif "error" in result:
                status = "ERROR"
            elif name == "run_public_tests":
                status = "PASS" if result.get("passed") else "FAIL"
            elif name == "write_solution":
                status = "WRITTEN" if result.get("status") == "written" else "UNKNOWN"
            else:
                status = "OK"
            events.append((label, status, raw, json.dumps(result, indent=2, ensure_ascii=False)))

        # Compress only consecutive identical actions and results.
        for _, group in groupby(events):
            batch = list(group)
            label, status, raw, result = batch[0]
            count = f" ×{len(batch)}" if len(batch) > 1 else ""
            color = "#b91c1c" if status in {"FAIL", "ERROR", "INVALID JSON"} else "#15803d"
            lines.append(
                "<details style='margin:8px 0 8px 20px'>"
                f"<summary><code>{escape(label)}</code>{count} "
                f"<strong style='color:{color}'>{escape(status)}</strong></summary>"
                f"<pre style='white-space:pre-wrap;overflow-wrap:anywhere'>"
                f"Arguments:\n{escape(raw)}\n\nResult:\n{escape(result)}</pre></details>"
            )
        if not calls:
            lines.append("<p>No tool calls: model finished.</p>")
        lines.append("</details>")

    lines.append(
        "<details><summary><b>Final solution.py</b></summary>"
        f"<pre>{escape((workspace / 'solution.py').read_text())}</pre></details>"
    )
    display(HTML("\n".join(lines)))


show_agent_trace(messages, usage_by_step)


## 5 — Independent hidden evaluation

This evaluation is outside the agent workspace; its tests were **not included in the model's tool outputs**. The code is copied to a separate temporary evaluation directory.

In [8]:
hidden_tests = """import unittest
from solution import median

class HiddenTests(unittest.TestCase):
    def test_negative_even(self):
        self.assertEqual(median([-7, -1, 3, 5]), 1)

    def test_duplicates(self):
        self.assertEqual(median([9, 1, 1, 1]), 1)

    def test_float(self):
        self.assertEqual(median([0.1, 0.3]), 0.2)

    def test_single(self):
        self.assertEqual(median([42]), 42)

    def test_even_unsorted(self):
        self.assertEqual(median([8, 1, 2, 7]), 4.5)

    def test_empty(self):
        with self.assertRaises(ValueError):
            median([])
"""

eval_dir = Path.cwd() / "agent-hidden-eval"
eval_dir.mkdir(parents=True, exist_ok=True)


shutil.copy2(workspace / "solution.py", eval_dir / "solution.py")
(eval_dir / "test_hidden.py").write_text(hidden_tests, encoding="utf-8")

public = run_tests(workspace, "test_public.py")
hidden = run_tests(eval_dir, "test_hidden.py")
print("Public PASS:", public["passed"])
print("Hidden PASS:", hidden["passed"])
print("Hidden details:", hidden["output"])
print("Agent steps:", min(step, MAX_STEPS))

Public PASS: True
Hidden PASS: True
Hidden details: test_duplicates (test_hidden.HiddenTests.test_duplicates) ... ok
test_empty (test_hidden.HiddenTests.test_empty) ... ok
test_even_unsorted (test_hidden.HiddenTests.test_even_unsorted) ... ok
test_float (test_hidden.HiddenTests.test_float) ... ok
test_negative_even (test_hidden.HiddenTests.test_negative_even) ... ok
test_single (test_hidden.HiddenTests.test_single) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.000s

OK

Agent steps: 2
